# CSRNet + Mall: continue training on a Colab GPU
This notebook resumes the completed Mall run toward **10 total epochs**, with **batch size 4** and **Adam 1e-5**.

In VS Code, choose **Select Kernel → Colab → your T4 server**, then run all cells in order. In the Colab website, choose a GPU runtime first. The first cell tests the actual CUDA device.

The existing `mall_csrnet_bundle.zip` in My Drive is reused. Current Python scripts are embedded below, so no separate script upload or new dataset ZIP is needed.

Your original run `20260903_010838_885145_UTC` stays intact. Continuation results go into the separate `20260903_010838_885145_UTC_continued` folder. Missing checkpoints cause a clear error rather than silently starting over.

At your previous T4 speed, nine more epochs take about 39 minutes plus setup and checkpoint saving. Results are selected using validation MAE; the test pass runs once at the end.


## 1. Check the GPU and essential packages
Keep Colab's supplied **torch + torchvision** pair. Do not install the laptop's `uv.lock`, a CPU-only PyTorch wheel, or `pip install -e .` here.

If the code cell below reports no CUDA GPU, change the runtime type before continuing. GPU memory is reported so any memory limitation is visible; batch size is never changed automatically.

In [1]:
import importlib.util
import importlib.metadata
import subprocess
import sys
from pathlib import Path

import torch

if not Path("/content").is_dir():
    raise RuntimeError("Select the remote Colab kernel in VS Code, then rerun this cell.")

if sys.version_info < (3, 11):
    raise RuntimeError("This project requires Python 3.11 or newer.")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. In VS Code select Kernel > Colab > a GPU server; "
        "then rerun. This notebook will not start CPU training."
    )

# Install only absent essentials, or upgrade an old Pillow lacking the font API.
packages = []
for module, package in (("numpy", "numpy"), ("h5py", "h5py>=3.8")):
    if importlib.util.find_spec(module) is None:
        packages.append(package)
try:
    pillow_version = importlib.metadata.version("Pillow")
    if tuple(int(part) for part in pillow_version.split(".")[:2]) < (10, 1):
        packages.append("Pillow>=10.1")
except importlib.metadata.PackageNotFoundError:
    packages.append("Pillow>=10.1")
if packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

import torchvision
import h5py
import numpy
import PIL

# Allocate on CUDA and synchronize: kernel metadata alone does not prove GPU access.
probe = torch.ones(4, device="cuda")
assert float((probe * 2).sum().item()) == 8.0
torch.cuda.synchronize()
del probe
gpu = torch.cuda.get_device_properties(0)
print("Python:", sys.version.split()[0])
print("Kernel interpreter:", sys.executable)
print("CUDA available:", torch.cuda.is_available(), "| PyTorch CUDA build:", torch.version.cuda)
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU:", gpu.name)
print(f"GPU memory: {gpu.total_memory / 1024**3:.1f} GiB")
print("Batch size remains 4. No CPU fallback or automatic batch-size reduction.")

## 2. Mount Drive, copy the ZIP, and verify the unchanged files
The default upload location is `My Drive/mall_csrnet_bundle.zip`. Edit `BUNDLE_ON_DRIVE` only if you uploaded it somewhere else.

This step copies the archive to Colab, checks its SHA-256, safely extracts it, and verifies every file against the manifest. Allow a few minutes for copying/extraction. It does not move or delete your Drive upload.

In [2]:
import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
BUNDLE_ON_DRIVE = DRIVE_ROOT / "mall_csrnet_bundle.zip"
EXPECTED_BUNDLE_SHA256 = "3088c200659757144c6e35c38bcf9c3afc9cf0ede1b9cdcc590fee0c1c8f73bc"

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

if not BUNDLE_ON_DRIVE.is_file():
    raise FileNotFoundError(
        f"Upload mall_csrnet_bundle.zip to My Drive first. Missing: {BUNDLE_ON_DRIVE}"
    )

LOCAL_BUNDLE = Path("/content") / f"mall_csrnet_{EXPECTED_BUNDLE_SHA256[:12]}.zip"
if not LOCAL_BUNDLE.exists():
    print("Copying the completed Drive upload onto Colab's local disk...", flush=True)
    shutil.copyfile(BUNDLE_ON_DRIVE, LOCAL_BUNDLE)
if sha256_file(LOCAL_BUNDLE) != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        "The ZIP checksum does not match this notebook. Do not train with it. "
        "Use the matching original notebook/ZIP pair and a fresh runtime."
    )

PROJECT_DIR = Path("/content") / f"mall_csrnet_verified_{EXPECTED_BUNDLE_SHA256[:12]}"
marker = PROJECT_DIR / ".verified_bundle_sha256"
if PROJECT_DIR.exists() and (
    not marker.is_file() or marker.read_text().strip() != EXPECTED_BUNDLE_SHA256
):
    raise RuntimeError(
        "An incomplete or different extraction already exists. Use a fresh runtime; "
        "this cell will not overwrite an unverified project."
    )

with zipfile.ZipFile(LOCAL_BUNDLE) as archive:
    manifest = json.loads(archive.read("bundle_manifest.json"))
    names = archive.namelist()
    expected_names = set(manifest["files"]) | {"bundle_manifest.json"}
    if len(names) != len(set(names)) or set(names) != expected_names:
        raise RuntimeError("Archive contents do not match the manifest.")
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        mode = member.external_attr >> 16
        if (relative.is_absolute() or ".." in relative.parts
                or "\\" in member.filename or stat.S_ISLNK(mode)
                or not (PROJECT_DIR / member.filename).resolve().is_relative_to(PROJECT_DIR.resolve())):
            raise RuntimeError(f"Unsafe archive member: {member.filename}")
    if not PROJECT_DIR.exists():
        PROJECT_DIR.mkdir()
        archive.extractall(PROJECT_DIR)

for index, (name, details) in enumerate(manifest["files"].items(), 1):
    path = PROJECT_DIR / name
    if path.stat().st_size != details["bytes"] or sha256_file(path) != details["sha256"]:
        raise RuntimeError(f"Uploaded file failed integrity verification: {name}")
    if index % 1000 == 0:
        print(f"Verified {index}/{len(manifest['files'])} files", flush=True)

frames_dir = PROJECT_DIR / "cv_project/mall_dataset/frames/frames"
density_dir = PROJECT_DIR / "cv_project/mall_dataset/density_maps"
assert {p.name for p in frames_dir.glob("*.jpg")} == {
    f"seq_{i:06d}.jpg" for i in range(1, 2001)
}
assert {p.name for p in density_dir.glob("*.h5")} == {
    f"frame_{i:04d}.h5" for i in range(1, 2001)
}
assert manifest["batch_size"] == 4
assert manifest["splits"] == {
    "train": [1, 1400], "validation": [1401, 1700], "test": [1701, 2000]
}
marker.write_text(EXPECTED_BUNDLE_SHA256 + "\n", encoding="utf-8")
print("Verified 2,000 unchanged JPG/HDF5 pairs.")
print("Local project:", PROJECT_DIR)
BUNDLE_PROJECT_DIR = PROJECT_DIR


## 3. Load current scripts and prepare the continuation
The collapsed cell contains the reviewed Python source snapshot. The next cell stages it beside the verified data and runs the tests.

The first execution copies your previous best checkpoint into a separate continuation folder. Later executions select the latest completed checkpoint there. `EPOCHS = 10` is the **total**, including epoch 1. Leave `SKIP_TEST = False` for one final test evaluation after training.


In [3]:
# Current source snapshot; refresh after editing the Python scripts.
NOTEBOOK_SOURCES = {
  "csrnet.py": "\"\"\"CSRNet: VGG16 conv1_1 through conv4_3, then a dilated density backend.\n\nArchitecture reference: https://github.com/leeyeehoo/CSRNet-pytorch\nThe classification head, pool4, and pool5 are NOT part of this model.\n\"\"\"\n\nimport torch\nfrom torch import nn\nfrom torchvision.models import VGG16_Weights\n\n\ndef make_layers(channels: list[int | str], in_channels: int, dilation: int = 1) -> nn.Sequential:\n    layers: list[nn.Module] = []\n    for out_channels in channels:\n        if out_channels == \"M\":\n            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))\n        else:\n            layers.extend([\n                nn.Conv2d(in_channels, int(out_channels), 3, padding=dilation, dilation=dilation),\n                nn.ReLU(inplace=True),\n            ])\n            in_channels = int(out_channels)\n    return nn.Sequential(*layers)\n\n\nclass CSRNet(nn.Module):\n    def __init__(self, pretrained: bool = True) -> None:\n        super().__init__()\n        self.frontend = make_layers(\n            [64, 64, \"M\", 128, 128, \"M\", 256, 256, 256, \"M\", 512, 512, 512], 3\n        )\n        self.backend = make_layers([512, 512, 512, 256, 128, 64], 512, dilation=2)\n        self.output_layer = nn.Conv2d(64, 1, kernel_size=1)\n        self.pretrained_frontend = pretrained\n\n        for layer in self.modules():\n            if isinstance(layer, nn.Conv2d):\n                nn.init.normal_(layer.weight, std=0.01)\n                nn.init.zeros_(layer.bias)\n\n        if pretrained:\n            # Load only matching frontend parameters. No classifier is instantiated.\n            weights = VGG16_Weights.IMAGENET1K_V1.get_state_dict(progress=True, check_hash=True)\n            frontend_weights = {\n                name: weights[f\"features.{name}\"] for name in self.frontend.state_dict()\n            }\n            self.frontend.load_state_dict(frontend_weights, strict=True)\n\n    def forward(self, images: torch.Tensor) -> torch.Tensor:\n        # Standard CSRNet has a linear output, not a final ReLU or classifier.\n        return self.output_layer(self.backend(self.frontend(images)))\n",
  "density_utils.py": "\"\"\"Count-preserving target resizing; no image or on-disk data changes.\"\"\"\n\nimport torch\nfrom torch.nn import functional as F\n\n\ndef density_counts(density: torch.Tensor) -> torch.Tensor:\n    \"\"\"One count per sample; accumulate in float64 to reduce summation error.\"\"\"\n    return density.sum(dim=(1, 2, 3), dtype=torch.float64)\n\n\ndef resize_density_preserving_count(\n    target: torch.Tensor, size: tuple[int, int]\n) -> torch.Tensor:\n    \"\"\"Area-average, compensate pixel area, and preserve each sample's mass.\n\n    For 480x640 -> 60x80, every output pixel sums an 8x8 input block.\n    For non-integer ratios, per-sample correction also removes the mass error\n    caused by overlapping adaptive-area bins. Empty density maps stay zero.\n    \"\"\"\n    if target.ndim != 4 or target.shape[1] != 1 or not target.is_floating_point():\n        raise ValueError(\"Expected floating-point density targets shaped (B, 1, H, W)\")\n    height, width = target.shape[-2:]\n    out_height, out_width = size\n    if not (0 < out_height <= height and 0 < out_width <= width):\n        raise ValueError(\"Target size must be positive and no larger than the input\")\n    if not torch.isfinite(target).all() or (target < 0).any():\n        raise ValueError(\"Ground-truth density must be finite and non-negative\")\n    if (height, width) == size:\n        return target\n\n    resized = F.interpolate(target, size=size, mode=\"area\")\n    resized = resized * (height * width / (out_height * out_width))\n    original_counts = density_counts(target)\n    resized_counts = density_counts(resized)\n    if ((original_counts > 0) & (resized_counts <= 0)).any():\n        raise RuntimeError(\"Resizing lost all density in a non-empty target\")\n\n    # No epsilon is added to the true counts: zero and tiny-count maps stay valid.\n    scale = original_counts / torch.where(resized_counts > 0, resized_counts, 1.0)\n    resized = resized * scale.to(resized.dtype).view(-1, 1, 1, 1)\n    if not torch.allclose(density_counts(resized), original_counts, atol=1e-4, rtol=1e-6):\n        raise RuntimeError(\"Density counts were not preserved; refusing to train\")\n    return resized\n",
  "train_csrnet.py": "\"\"\"Train CSRNet on the existing Mall splits; select by validation count MAE.\n\nFresh run: uv run python train_csrnet.py --epochs 1 --lr 1e-5\nResume: uv run python train_csrnet.py --epochs 10 --lr 1e-5 --resume CHECKPOINT\nNo Dataset, image file, annotation, density file, or split is modified.\n\"\"\"\n\nimport argparse\nimport json\nimport logging\nimport math\nimport os\nimport sys\nimport time\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw, ImageFont\nfrom torch import nn\nfrom torch.utils.data import DataLoader\n\nfrom csrnet import CSRNet\nfrom density_utils import density_counts, resize_density_preserving_count\nfrom mall_dataloader import test_loader, train_loader, validation_loader\n\n\nPROJECT_ROOT = Path(__file__).resolve().parent\nLOG = logging.getLogger(__name__)\n\n\ndef atomic_write_json(path: Path, value: dict) -> None:\n    \"\"\"Replace a JSON file only after its complete new contents are written.\"\"\"\n    temporary = path.with_suffix(path.suffix + \".tmp\")\n    temporary.write_text(json.dumps(value, indent=2), encoding=\"utf-8\")\n    temporary.replace(path)\n\n\ndef atomic_save_checkpoint(payload: dict, path: Path) -> None:\n    \"\"\"Keep the previous checkpoint if writing its replacement fails.\"\"\"\n    temporary = path.with_suffix(path.suffix + \".tmp\")\n    with temporary.open(\"wb\") as stream:\n        torch.save(payload, stream)\n        stream.flush()\n        os.fsync(stream.fileno())\n    temporary.replace(path)\n\n\ndef checkpoint_payload(\n    epoch: int,\n    model: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    validation: dict[str, float],\n    config: dict,\n    history: list[dict],\n    best_epoch: int,\n    best_mae: float,\n) -> dict:\n    \"\"\"Collect everything needed to resume after a completed epoch.\"\"\"\n    payload = {\n        \"epoch\": epoch,\n        \"model_state_dict\": model.state_dict(),\n        \"optimizer_state_dict\": optimizer.state_dict(),\n        \"validation_loss\": validation[\"loss\"],\n        \"validation_mae\": validation[\"mae\"],\n        \"validation_rmse\": validation[\"rmse\"],\n        \"training_config\": config,\n        \"history\": history,\n        \"best_epoch\": best_epoch,\n        \"best_validation_mae\": best_mae,\n        \"torch_rng_state\": torch.get_rng_state(),\n    }\n    if torch.cuda.is_available():\n        payload[\"cuda_rng_state_all\"] = torch.cuda.get_rng_state_all()\n    return payload\n\n\ndef validate_resume_config(saved: dict, current: dict) -> None:\n    \"\"\"Refuse to mix a checkpoint with a different training setup.\"\"\"\n    keys = (\n        \"model\", \"learning_rate\", \"optimizer\", \"loss\", \"pretrained_vgg16\",\n        \"weights\", \"seed\", \"batch_size\", \"parameters\", \"splits\", \"input_resolution\",\n        \"output_resolution\",\n    )\n    differences = [\n        key for key in keys\n        if key in saved and key in current and saved[key] != current[key]\n    ]\n    if differences:\n        raise ValueError(\n            \"Resume checkpoint does not match the current configuration: \"\n            + \", \".join(differences)\n        )\n\n\ndef load_resume_checkpoint(\n    path: Path,\n    model: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    config: dict,\n) -> tuple[int, int, float, list[dict]]:\n    \"\"\"Restore model, optimizer, history, and RNG state from a checkpoint.\"\"\"\n    checkpoint = torch.load(path, map_location=device, weights_only=True)\n    validate_resume_config(checkpoint.get(\"training_config\", {}), config)\n    model.load_state_dict(checkpoint[\"model_state_dict\"])\n    optimizer.load_state_dict(checkpoint[\"optimizer_state_dict\"])\n\n    completed_epoch = int(checkpoint[\"epoch\"])\n    best_epoch = int(checkpoint.get(\"best_epoch\", completed_epoch))\n    best_mae = float(\n        checkpoint.get(\"best_validation_mae\", checkpoint[\"validation_mae\"])\n    )\n    history = list(checkpoint.get(\"history\", []))\n    if \"torch_rng_state\" in checkpoint:\n        torch.set_rng_state(checkpoint[\"torch_rng_state\"].cpu())\n    else:\n        LOG.warning(\"Older checkpoint has no saved RNG state; continuation is valid but not exact.\")\n    if device.type == \"cuda\" and \"cuda_rng_state_all\" in checkpoint:\n        torch.cuda.set_rng_state_all(\n            [state.cpu() for state in checkpoint[\"cuda_rng_state_all\"]]\n        )\n    return completed_epoch + 1, best_epoch, best_mae, history\n\n\ndef normalize_images(images: torch.Tensor, pretrained: bool) -> torch.Tensor:\n    \"\"\"Only ImageNet channel normalization; no classification resize or crop.\"\"\"\n    if not pretrained:\n        return images\n    mean = images.new_tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)\n    std = images.new_tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)\n    return (images - mean) / std\n\n\ndef predict_and_match(\n    model: nn.Module, images: torch.Tensor, targets: torch.Tensor, pretrained: bool\n) -> tuple[torch.Tensor, torch.Tensor]:\n    predictions = model(normalize_images(images, pretrained))\n    resized = resize_density_preserving_count(targets, tuple(predictions.shape[-2:]))\n    if predictions.shape != resized.shape:\n        raise RuntimeError(f\"Prediction {predictions.shape} != target {resized.shape}\")\n    return predictions, resized\n\n\ndef count_error(original: torch.Tensor, resized: torch.Tensor) -> float:\n    return float((density_counts(original) - density_counts(resized)).abs().max().item())\n\n\ndef preflight(model: nn.Module, device: torch.device, pretrained: bool) -> dict:\n    \"\"\"Inspect a real full-size training batch before allowing optimization.\"\"\"\n    images, targets = next(iter(train_loader))\n    images, targets = images.to(device), targets.to(device)\n    model.eval()\n    started = time.perf_counter()\n    with torch.no_grad():\n        predictions, resized = predict_and_match(model, images, targets, pretrained)\n    if not torch.isfinite(predictions).all():\n        raise FloatingPointError(\"Non-finite preflight prediction\")\n    error = count_error(targets, resized)\n    details = {\n        \"input_shape\": list(images.shape),\n        \"original_target_shape\": list(targets.shape),\n        \"prediction_shape\": list(predictions.shape),\n        \"resized_target_shape\": list(resized.shape),\n        \"original_counts\": density_counts(targets).tolist(),\n        \"resized_counts\": density_counts(resized).tolist(),\n        \"max_count_error\": error,\n        \"forward_seconds\": time.perf_counter() - started,\n    }\n    for name, value in details.items():\n        LOG.info(\"Preflight %s: %s\", name, value)\n    return details\n\n\ndef train_one_epoch(\n    model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer,\n    criterion: nn.Module, device: torch.device, pretrained: bool,\n) -> dict[str, float]:\n    model.train()\n    total_loss, samples, max_error = 0.0, 0, 0.0\n    started = time.perf_counter()\n    for step, (images, original_targets) in enumerate(loader, 1):\n        images, original_targets = images.to(device), original_targets.to(device)\n        predictions, targets = predict_and_match(model, images, original_targets, pretrained)\n        loss = criterion(predictions, targets)\n        if not torch.isfinite(loss):\n            raise FloatingPointError(f\"Non-finite training loss at batch {step}\")\n        optimizer.zero_grad(set_to_none=True)\n        loss.backward()\n        optimizer.step()\n\n        size = images.shape[0]\n        total_loss += loss.item() * size\n        samples += size\n        max_error = max(max_error, count_error(original_targets, targets))\n        if step == 1 or step % 10 == 0 or step == len(loader):\n            elapsed = time.perf_counter() - started\n            LOG.info(\n                \"Train %d/%d | mean MSE %.8f | %.2f s/batch | estimated train %.1f min\",\n                step, len(loader), total_loss / samples, elapsed / step,\n                elapsed / step * len(loader) / 60,\n            )\n    return {\"loss\": total_loss / samples, \"max_count_error\": max_error,\n            \"seconds\": time.perf_counter() - started}\n\n\ndef save_prediction(\n    image: torch.Tensor, target: torch.Tensor, prediction: torch.Tensor,\n    true_count: float, path: Path,\n) -> None:\n    \"\"\"Three plain panels; low-resolution maps are enlarged for DISPLAY only.\"\"\"\n    rgb = (image.detach().cpu().permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)\n    gt = target.detach().cpu().squeeze(0).numpy()\n    pred = prediction.detach().cpu().squeeze(0).numpy()\n    limit = max(float(np.abs(gt).max()), float(np.abs(pred).max()), 1e-12)\n    height, width = rgb.shape[:2]\n\n    def heatmap(values: np.ndarray) -> Image.Image:\n        scaled = np.clip(values / limit, -1, 1)\n        colors = np.zeros((*values.shape, 3), dtype=np.uint8)\n        colors[..., 0] = (255 * np.maximum(scaled, 0)).astype(np.uint8)\n        colors[..., 2] = (255 * np.maximum(-scaled, 0)).astype(np.uint8)\n        return Image.fromarray(colors).resize((width, height), Image.Resampling.NEAREST)\n\n    canvas = Image.new(\"RGB\", (3 * width, height + 64), \"white\")\n    for index, panel in enumerate((Image.fromarray(rgb), heatmap(gt), heatmap(pred))):\n        canvas.paste(panel, (index * width, 64))\n    draw = ImageDraw.Draw(canvas)\n    font = ImageFont.load_default(size=17)\n    predicted_count = float(prediction.sum(dtype=torch.float64).item())\n    labels = (\n        \"Original RGB image\",\n        f\"Ground truth count: {true_count:.2f}\",\n        f\"Predicted count: {predicted_count:.2f}\",\n    )\n    for index, label in enumerate(labels):\n        draw.text((index * width + 10, 7), label, fill=\"black\", font=font)\n    draw.text((width + 10, 34), f\"Density @ {gt.shape}: shared scale\", fill=\"black\", font=font)\n    draw.text((2 * width + 10, 34), \"Red: positive | Blue: negative\", fill=\"black\", font=font)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(path)\n\n\ndef evaluate(\n    model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device,\n    pretrained: bool, name: str, prediction_dir: Path | None = None,\n) -> dict[str, float]:\n    model.eval()\n    total_loss, absolute_error, squared_error = 0.0, 0.0, 0.0\n    samples, max_error = 0, 0.0\n    with torch.no_grad():\n        for step, (images, original_targets) in enumerate(loader, 1):\n            images, original_targets = images.to(device), original_targets.to(device)\n            predictions, targets = predict_and_match(model, images, original_targets, pretrained)\n            loss = criterion(predictions, targets)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f\"Non-finite {name} loss at batch {step}\")\n            # Evaluation ground truth ALWAYS comes from the original full-resolution map.\n            true_counts = density_counts(original_targets)\n            errors = density_counts(predictions) - true_counts\n            size = images.shape[0]\n            total_loss += loss.item() * size\n            absolute_error += errors.abs().sum().item()\n            squared_error += errors.square().sum().item()\n            max_error = max(max_error, count_error(original_targets, targets))\n\n            if prediction_dir is not None and samples < 3:\n                for index in range(min(size, 3 - samples)):\n                    save_prediction(images[index], targets[index], predictions[index],\n                                    true_counts[index].item(),\n                                    prediction_dir / f\"test_sample_{samples + index + 1:03d}.png\")\n            samples += size\n            if step == 1 or step % 25 == 0 or step == len(loader):\n                LOG.info(\"%s %d/%d\", name, step, len(loader))\n    result = {\"loss\": total_loss / samples, \"mae\": absolute_error / samples,\n              \"rmse\": math.sqrt(squared_error / samples), \"max_count_error\": max_error}\n    if not all(math.isfinite(value) for value in result.values()):\n        raise FloatingPointError(f\"Non-finite {name} metrics\")\n    return result\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--epochs\", type=int, default=1)\n    parser.add_argument(\"--lr\", type=float, default=1e-5)\n    parser.add_argument(\"--seed\", type=int, default=42)\n    parser.add_argument(\"--no-pretrained\", action=\"store_true\",\n                        help=\"Explicitly opt out of VGG16 weights and ImageNet normalization\")\n    parser.add_argument(\"--output-dir\", type=Path, default=PROJECT_ROOT / \"outputs\")\n    parser.add_argument(\n        \"--resume\", type=Path,\n        help=\"Resume from a completed-epoch checkpoint; --epochs is the total epoch count\",\n    )\n    parser.add_argument(\n        \"--skip-test\", action=\"store_true\",\n        help=\"Finish after validation; useful before the final test evaluation\",\n    )\n    parser.add_argument(\"--require-cuda\", action=\"store_true\",\n                        help=\"Stop instead of falling back to CPU training\")\n    parser.add_argument(\"--preflight-only\", action=\"store_true\",\n                        help=\"Verify real batch shapes/counts without training or test evaluation\")\n    args = parser.parse_args()\n    if args.epochs < 1 or not math.isfinite(args.lr) or args.lr <= 0:\n        parser.error(\"epochs and lr must be positive\")\n    return args\n\n\ndef main() -> None:\n    args = parse_args()\n    checkpoint_path = args.output_dir / \"checkpoints\" / \"csrnet_best.pt\"\n    latest_checkpoint_path = args.output_dir / \"checkpoints\" / \"csrnet_latest.pt\"\n    report_path = args.output_dir / \"csrnet_results.json\"\n    if args.resume is None and not args.preflight_only and any(\n        path.exists() for path in (checkpoint_path, latest_checkpoint_path, report_path)\n    ):\n        raise FileExistsError(\"This output folder already contains a run. Use --resume or a new folder.\")\n    if args.resume is not None:\n        args.resume = args.resume.expanduser().resolve()\n        if not args.resume.is_file():\n            raise FileNotFoundError(f\"Resume checkpoint not found: {args.resume}\")\n        if args.resume.parent != checkpoint_path.parent.resolve():\n            raise ValueError(\"Resume checkpoint must be in --output-dir/checkpoints; copy the run first.\")\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    run_id = datetime.now().strftime(\"%Y%m%d_%H%M%S\")\n    log_path = args.output_dir / f\"csrnet_{run_id}.log\"\n    logging.basicConfig(level=logging.INFO, format=\"%(message)s\", force=True, handlers=[\n        logging.StreamHandler(sys.stdout), logging.FileHandler(log_path, encoding=\"utf-8\"),\n    ])\n    torch.manual_seed(args.seed)\n    torch.hub.set_dir(str(args.output_dir / \"torch_cache\"))\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    if args.require_cuda and device.type != \"cuda\":\n        raise RuntimeError(\"CUDA is required for this run; select the Colab GPU kernel.\")\n    pretrained = not args.no_pretrained\n    LOG.info(\"Device: %s | pretrained requested: %s | batch size: %s | lr: %g | epochs: %d\",\n             device, pretrained, train_loader.batch_size, args.lr, args.epochs)\n    # A failed download raises an error; there is NO silent random-weight fallback.\n    # A resumed checkpoint supplies the whole model, including its VGG frontend.\n    model = CSRNet(pretrained=pretrained and args.resume is None).to(device)\n    parameters = sum(parameter.numel() for parameter in model.parameters())\n    LOG.info(\"Pretrained VGG16 used: %s | trainable parameters: %d\", pretrained, parameters)\n    verified = preflight(model, device, pretrained)\n    if args.preflight_only:\n        LOG.info(\"Preflight passed; no training or test evaluation requested.\")\n        return\n\n    criterion = nn.MSELoss()\n    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)\n    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)\n    config = {\n        \"model\": \"CSRNet\", \"epochs\": args.epochs, \"learning_rate\": args.lr,\n        \"optimizer\": \"Adam\", \"loss\": \"MSELoss(mean)\", \"seed\": args.seed,\n        \"pretrained_vgg16\": pretrained,\n        \"weights\": \"VGG16_Weights.IMAGENET1K_V1\" if pretrained else None,\n        \"imagenet_mean\": [0.485, 0.456, 0.406] if pretrained else None,\n        \"imagenet_std\": [0.229, 0.224, 0.225] if pretrained else None,\n        \"device\": str(device), \"batch_size\": train_loader.batch_size,\n        \"parameters\": parameters, \"torch_version\": str(torch.__version__),\n        \"splits\": {\"train\": [1, 1400], \"validation\": [1401, 1700], \"test\": [1701, 2000]},\n        \"input_resolution\": verified[\"input_shape\"][-2:],\n        \"output_resolution\": verified[\"prediction_shape\"][-2:],\n    }\n    history: list[dict] = []\n    best_epoch = 0\n    best_mae = math.inf\n    start_epoch = 1\n    if args.resume is not None:\n        resume_path = args.resume.expanduser().resolve()\n        if not resume_path.is_file():\n            raise FileNotFoundError(f\"Resume checkpoint not found: {resume_path}\")\n        start_epoch, best_epoch, best_mae, history = load_resume_checkpoint(\n            resume_path, model, optimizer, device, config\n        )\n        # The original one-epoch checkpoint predates embedded history. Recover it\n        # from the adjacent result JSON when resuming inside the same run folder.\n        previous_report = resume_path.parent.parent / \"csrnet_results.json\"\n        if not history and previous_report.is_file():\n            history = [entry for entry in json.loads(\n                previous_report.read_text(encoding=\"utf-8\")\n            ).get(\"epochs\", []) if entry[\"epoch\"] < start_epoch]\n        if not checkpoint_path.exists():\n            raise FileNotFoundError(\n                \"The run's best checkpoint is missing. Resume in its original --output-dir.\"\n            )\n        saved_best = torch.load(checkpoint_path, map_location=\"cpu\", weights_only=True)\n        validate_resume_config(saved_best.get(\"training_config\", {}), config)\n        if saved_best[\"epoch\"] >= start_epoch:\n            raise ValueError(\"Best checkpoint is newer than the resume point; resume from that best checkpoint.\")\n        best_epoch = int(saved_best[\"epoch\"])\n        best_mae = float(saved_best[\"validation_mae\"])\n        del saved_best\n        LOG.info(\n            \"Resumed completed epoch %d; continuing at epoch %d toward total epoch %d\",\n            start_epoch - 1, start_epoch, args.epochs,\n        )\n    if start_epoch > args.epochs:\n        raise ValueError(\n            f\"Checkpoint already completed epoch {start_epoch - 1}; \"\n            f\"--epochs must be at least {start_epoch}.\"\n        )\n\n    report_path = args.output_dir / \"csrnet_results.json\"\n    for epoch in range(start_epoch, args.epochs + 1):\n        LOG.info(\"Epoch %d/%d\", epoch, args.epochs)\n        started = time.perf_counter()\n        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device, pretrained)\n        validation = evaluate(model, validation_loader, criterion, device, pretrained, \"Validation\")\n        seconds = time.perf_counter() - started\n        history.append({\"epoch\": epoch, \"train\": train_metrics, \"validation\": validation,\n                        \"epoch_seconds\": seconds})\n        LOG.info(\"Train Loss: %.8f | Validation Loss: %.8f | Validation MAE: %.6f | \"\n                 \"Validation RMSE: %.6f | Epoch Time: %.1f s\", train_metrics[\"loss\"],\n                 validation[\"loss\"], validation[\"mae\"], validation[\"rmse\"], seconds)\n        if validation[\"mae\"] < best_mae:\n            best_mae = validation[\"mae\"]\n            best_epoch = epoch\n            atomic_save_checkpoint(\n                checkpoint_payload(\n                    epoch, model, optimizer, validation, config, history,\n                    best_epoch, best_mae,\n                ),\n                checkpoint_path,\n            )\n            LOG.info(\"Saved best validation checkpoint: %s\", checkpoint_path)\n\n        atomic_save_checkpoint(\n            checkpoint_payload(\n                epoch, model, optimizer, validation, config, history,\n                best_epoch, best_mae,\n            ),\n            latest_checkpoint_path,\n        )\n        LOG.info(\"Saved latest resumable checkpoint: %s\", latest_checkpoint_path)\n        atomic_write_json(\n            report_path,\n            {\n                \"status\": \"training_complete\" if epoch == args.epochs else \"training\",\n                \"config\": config,\n                \"preflight\": verified,\n                \"epochs\": history,\n                \"best_epoch\": best_epoch,\n                \"test\": None,\n                \"checkpoint_path\": str(checkpoint_path),\n                \"latest_checkpoint_path\": str(latest_checkpoint_path),\n                \"prediction_dir\": None,\n                \"log_path\": str(log_path),\n            },\n        )\n\n    if args.skip_test:\n        LOG.info(\"Training complete; test evaluation skipped as requested.\")\n        return\n\n    # Exactly one final test pass, using weights chosen only by validation MAE.\n    best = torch.load(checkpoint_path, map_location=device, weights_only=True)\n    model.load_state_dict(best[\"model_state_dict\"])\n    LOG.info(\"Loaded best epoch %d for the single test evaluation\", best[\"epoch\"])\n    prediction_dir = args.output_dir / \"csrnet_predictions\"\n    test = evaluate(model, test_loader, criterion, device, pretrained, \"Test\", prediction_dir)\n    LOG.info(\"Test MSE: %.8f | Test MAE: %.6f | Test RMSE: %.6f\",\n             test[\"loss\"], test[\"mae\"], test[\"rmse\"])\n    report = {\"status\": \"complete\", \"config\": config, \"preflight\": verified, \"epochs\": history,\n              \"best_epoch\": best[\"epoch\"], \"test\": test,\n              \"checkpoint_path\": str(checkpoint_path), \"prediction_dir\": str(prediction_dir),\n              \"latest_checkpoint_path\": str(latest_checkpoint_path),\n              \"log_path\": str(log_path)}\n    atomic_write_json(report_path, report)\n    LOG.info(\"Completed. Results: %s | Predictions: %s\", report_path, prediction_dir)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "mall_torch_dataset.py": "\"\"\"Minimal PyTorch Dataset for paired Mall images and density maps.\"\"\"\n\nfrom pathlib import Path\n\nimport h5py\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\n\nPROJECT_ROOT = Path(__file__).resolve().parent\nDEFAULT_DATASET_DIR = PROJECT_ROOT / \"cv_project\" / \"mall_dataset\"\n\n\nclass MallDataset(Dataset):\n    def __init__(self, dataset_dir: Path = DEFAULT_DATASET_DIR) -> None:\n        images_dir = Path(dataset_dir) / \"frames\" / \"frames\"\n        density_dir = Path(dataset_dir) / \"density_maps\"\n\n        self.samples: list[tuple[Path, Path]] = []\n        for image_path in sorted(images_dir.glob(\"seq_*.jpg\")):\n            frame_number = int(image_path.stem.removeprefix(\"seq_\"))\n            density_path = density_dir / f\"frame_{frame_number:04d}.h5\"\n            if not density_path.exists():\n                raise FileNotFoundError(f\"Missing density map for {image_path.name}\")\n            self.samples.append((image_path, density_path))\n\n        if not self.samples:\n            raise FileNotFoundError(f\"No Mall JPG frames found in {images_dir}\")\n\n    def __len__(self) -> int:\n        return len(self.samples)\n\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:\n        image_path, density_path = self.samples[index]\n\n        with Image.open(image_path) as opened_image:\n            image_array = np.array(opened_image.convert(\"RGB\"), copy=True)\n\n        with h5py.File(density_path, \"r\") as h5_file:\n            density_array = h5_file[\"density\"][:]\n\n        image_tensor = (\n            torch.from_numpy(image_array).permute(2, 0, 1).to(torch.float32) / 255.0\n        )\n        density_tensor = torch.from_numpy(density_array).to(torch.float32).unsqueeze(0)\n        return image_tensor, density_tensor\n\n\nif __name__ == \"__main__\":\n    dataset = MallDataset()\n    image_tensor, density_tensor = dataset[0]\n\n    print(f\"Dataset length:       {len(dataset)}\")\n    print(f\"Image tensor shape:   {tuple(image_tensor.shape)}\")\n    print(f\"Image dtype:          {image_tensor.dtype}\")\n    print(f\"Image min/max:        {image_tensor.min().item():.6f} / {image_tensor.max().item():.6f}\")\n    print(f\"Density tensor shape: {tuple(density_tensor.shape)}\")\n    print(f\"Density dtype:        {density_tensor.dtype}\")\n    print(f\"Density sum:          {density_tensor.sum().item():.6f}\")\n",
  "mall_dataloader.py": "\"\"\"Create and inspect contiguous Mall Dataset train/validation/test splits.\"\"\"\n\nfrom torch.utils.data import DataLoader, Subset\n\nfrom mall_torch_dataset import MallDataset\n\n\ndataset = MallDataset()\n\ntrain_indices = range(0, 1400)\nvalidation_indices = range(1400, 1700)\ntest_indices = range(1700, 2000)\n\ntrain_dataset = Subset(dataset, train_indices)\nvalidation_dataset = Subset(dataset, validation_indices)\ntest_dataset = Subset(dataset, test_indices)\n\ntrain_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)\nvalidation_loader = DataLoader(validation_dataset, batch_size=4, shuffle=False)\ntest_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)\n\nsplit_total = len(train_dataset) + len(validation_dataset) + len(test_dataset)\nassert split_total == len(dataset) == 2000\n\nif __name__ == \"__main__\":\n    print(f\"Total dataset length: {len(dataset)}\")\n    print(f\"Train length:         {len(train_dataset)}\")\n    print(f\"Validation length:    {len(validation_dataset)}\")\n    print(f\"Test length:          {len(test_dataset)}\")\n    print(f\"Train source frames:      {train_indices.start + 1}-{train_indices.stop}\")\n    print(\n        \"Validation source frames: \"\n        f\"{validation_indices.start + 1}-{validation_indices.stop}\"\n    )\n    print(f\"Test source frames:       {test_indices.start + 1}-{test_indices.stop}\")\n    print(f\"Split total check:    {split_total}\")\n\n    for split_name, loader in (\n        (\"Train\", train_loader),\n        (\"Validation\", validation_loader),\n        (\"Test\", test_loader),\n    ):\n        image_batch, density_batch = next(iter(loader))\n        print(f\"{split_name} image batch shape:   {tuple(image_batch.shape)}\")\n        print(f\"{split_name} density batch shape: {tuple(density_batch.shape)}\")\n",
  "test_csrnet.py": "\"\"\"Focused offline tests for the CSRNet architecture and tensor interface.\"\"\"\n\nimport tempfile\nimport unittest\nimport json\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nimport torch\nfrom PIL import Image\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset\n\nfrom csrnet import CSRNet\nfrom density_utils import density_counts, resize_density_preserving_count\nfrom train_csrnet import (\n    atomic_write_json,\n    atomic_save_checkpoint,\n    checkpoint_payload,\n    evaluate,\n    load_resume_checkpoint,\n    normalize_images,\n    save_prediction,\n    train_one_epoch,\n    validate_resume_config,\n)\nimport train_csrnet\n\n\nclass DensityResizeTests(unittest.TestCase):\n    def test_mall_resolution_is_block_sum(self):\n        torch.manual_seed(0)\n        target = torch.rand(4, 1, 480, 640) * 0.001\n        resized = resize_density_preserving_count(target, (60, 80))\n        expected = target.reshape(4, 1, 60, 8, 80, 8).sum(dim=(3, 5))\n        torch.testing.assert_close(resized, expected)\n        torch.testing.assert_close(density_counts(resized), density_counts(target),\n                                   atol=1e-4, rtol=1e-6)\n\n    def test_non_integer_ratio_edges_zero_and_tiny_maps(self):\n        target = torch.zeros(4, 1, 17, 19)\n        target[0, 0, 0, 0] = 2\n        target[0, 0, -1, -1] = 3\n        target[1] = 0.2\n        target[3, 0, 8, 9] = 1e-10\n        original = target.clone()\n        resized = resize_density_preserving_count(target, (5, 7))\n        torch.testing.assert_close(density_counts(resized), density_counts(target),\n                                   atol=1e-12, rtol=1e-6)\n        self.assertEqual(float(resized[2].sum()), 0)\n        self.assertTrue(torch.equal(target, original))\n        self.assertTrue((resized >= 0).all())\n\n    def test_identity(self):\n        target = torch.ones(2, 1, 8, 10)\n        self.assertIs(resize_density_preserving_count(target, (8, 10)), target)\n\n    def test_invalid_inputs(self):\n        with self.assertRaises(ValueError):\n            resize_density_preserving_count(torch.ones(2, 8, 8), (4, 4))\n        for value in (-1.0, float(\"nan\")):\n            with self.assertRaises(ValueError):\n                resize_density_preserving_count(torch.full((1, 1, 8, 8), value), (4, 4))\n        with self.assertRaises(ValueError):\n            resize_density_preserving_count(torch.ones(1, 1, 8, 8), (9, 4))\n\n\nclass CSRNetTests(unittest.TestCase):\n    def test_architecture_and_stride(self):\n        model = CSRNet(pretrained=False).eval()\n        self.assertEqual(sum(isinstance(x, nn.MaxPool2d) for x in model.modules()), 3)\n        backend_convs = [x for x in model.backend if isinstance(x, nn.Conv2d)]\n        self.assertEqual([x.out_channels for x in backend_convs], [512, 512, 512, 256, 128, 64])\n        self.assertTrue(all(x.dilation == (2, 2) for x in backend_convs))\n        self.assertTrue(all(parameter.requires_grad for parameter in model.parameters()))\n        self.assertIsInstance(model.output_layer, nn.Conv2d)\n        with torch.no_grad():\n            output = model(torch.zeros(2, 3, 32, 40))\n        self.assertEqual(tuple(output.shape), (2, 1, 4, 5))\n\n    def test_normalization_is_image_only_and_out_of_place(self):\n        images = torch.zeros(2, 3, 8, 8)\n        normalized = normalize_images(images, pretrained=True)\n        expected = -torch.tensor([0.485, 0.456, 0.406]) / torch.tensor([0.229, 0.224, 0.225])\n        torch.testing.assert_close(normalized[0, :, 0, 0], expected)\n        self.assertEqual(float(images.sum()), 0)\n        self.assertIs(normalize_images(images, pretrained=False), images)\n\n    def test_count_metrics_and_no_evaluation_gradients(self):\n        class ConstantDensity(nn.Module):\n            def forward(self, images):\n                return images.new_full((images.shape[0], 1, 1, 1), 6.0)\n\n        images = torch.zeros(2, 3, 8, 8)\n        targets = torch.ones(2, 1, 8, 8) / 64\n        targets[0] *= 8\n        targets[1] *= 4\n        loader = DataLoader(TensorDataset(images, targets), batch_size=2)\n        metrics = evaluate(ConstantDensity(), loader, nn.MSELoss(), torch.device(\"cpu\"), False, \"Unit\")\n        self.assertAlmostEqual(metrics[\"loss\"], 4.0)\n        self.assertAlmostEqual(metrics[\"mae\"], 2.0)\n        self.assertAlmostEqual(metrics[\"rmse\"], 2.0)\n\n    def test_modular_training_step(self):\n        torch.manual_seed(1)\n        model = nn.Conv2d(3, 1, kernel_size=1, stride=8)\n        before = model.weight.detach().clone()\n        loader = DataLoader(TensorDataset(torch.ones(2, 3, 8, 8),\n                                         torch.ones(2, 1, 8, 8) / 64), batch_size=2)\n        metrics = train_one_epoch(model, loader, torch.optim.Adam(model.parameters(), lr=1e-5),\n                                  nn.MSELoss(), torch.device(\"cpu\"), False)\n        self.assertGreaterEqual(metrics[\"loss\"], 0)\n        self.assertFalse(torch.equal(before, model.weight))\n\n    def test_prediction_panel(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory) / \"prediction.png\"\n            save_prediction(torch.zeros(3, 64, 80), torch.ones(1, 8, 10),\n                            torch.full((1, 8, 10), -0.5), 80.0, path)\n            with Image.open(path) as saved:\n                self.assertEqual(saved.size, (240, 128))\n\n    def test_resume_checkpoint_round_trip(self):\n        torch.manual_seed(7)\n        model = nn.Conv2d(3, 1, kernel_size=1)\n        optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)\n        loss = model(torch.ones(1, 3, 2, 2)).square().mean()\n        loss.backward()\n        optimizer.step()\n        expected_weight = model.weight.detach().clone()\n        config = {\n            \"model\": \"test\", \"learning_rate\": 1e-5, \"optimizer\": \"Adam\",\n            \"loss\": \"MSELoss(mean)\", \"pretrained_vgg16\": False,\n            \"batch_size\": 4,\n        }\n        validation = {\"loss\": 0.5, \"mae\": 2.0, \"rmse\": 3.0}\n        history = [{\"epoch\": 1}]\n\n        with tempfile.TemporaryDirectory() as directory:\n            checkpoint = Path(directory) / \"latest.pt\"\n            torch.save(\n                checkpoint_payload(\n                    1, model, optimizer, validation, config, history, 1, 2.0\n                ),\n                checkpoint,\n            )\n            restored_model = nn.Conv2d(3, 1, kernel_size=1)\n            restored_optimizer = torch.optim.Adam(restored_model.parameters(), lr=1e-5)\n            start, best_epoch, best_mae, restored_history = load_resume_checkpoint(\n                checkpoint, restored_model, restored_optimizer,\n                torch.device(\"cpu\"), config,\n            )\n\n        self.assertEqual((start, best_epoch, best_mae), (2, 1, 2.0))\n        self.assertEqual(restored_history, history)\n        torch.testing.assert_close(restored_model.weight, expected_weight)\n        self.assertTrue(restored_optimizer.state_dict()[\"state\"])\n\n    def test_resume_rejects_changed_configuration(self):\n        with self.assertRaisesRegex(ValueError, \"learning_rate\"):\n            validate_resume_config(\n                {\"model\": \"CSRNet\", \"learning_rate\": 1e-5},\n                {\"model\": \"CSRNet\", \"learning_rate\": 2e-5},\n            )\n\n    def test_atomic_json_write(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory) / \"results.json\"\n            atomic_write_json(path, {\"status\": \"training\", \"epoch\": 2})\n            self.assertEqual(\n                path.read_text(encoding=\"utf-8\"),\n                '{\\n  \"status\": \"training\",\\n  \"epoch\": 2\\n}',\n            )\n            self.assertFalse(path.with_suffix(\".json.tmp\").exists())\n\n    def test_interrupted_checkpoint_save_preserves_previous_file(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory) / \"latest.pt\"\n            atomic_save_checkpoint({\"epoch\": 1}, path)\n            with patch(\"train_csrnet.torch.save\", side_effect=OSError(\"disk full\")):\n                with self.assertRaises(OSError):\n                    atomic_save_checkpoint({\"epoch\": 2}, path)\n            self.assertEqual(torch.load(path, weights_only=True)[\"epoch\"], 1)\n\n    def test_training_resume_matches_uninterrupted_run(self):\n        # Exercise the real main loop on tiny synthetic tensors. No Mall epoch runs.\n        images = torch.arange(6 * 3 * 8 * 8, dtype=torch.float32).reshape(6, 3, 8, 8) / 1200\n        targets = torch.ones(6, 1, 8, 8) / 64\n        dataset = TensorDataset(images, targets)\n        loader = DataLoader(dataset, batch_size=2, shuffle=True)\n        evaluation = DataLoader(dataset, batch_size=2)\n\n        def run(directory, epochs, resume=None):\n            argv = [\"train_csrnet.py\", \"--no-pretrained\", \"--epochs\", str(epochs),\n                    \"--output-dir\", str(directory), \"--skip-test\"]\n            if resume:\n                argv += [\"--resume\", str(resume)]\n            with patch(\"sys.argv\", argv):\n                train_csrnet.main()\n\n        with tempfile.TemporaryDirectory() as directory, \\\n                patch(\"train_csrnet.CSRNet\", side_effect=lambda **kwargs: nn.Conv2d(3, 1, 1, stride=8)), \\\n                patch(\"train_csrnet.train_loader\", loader), \\\n                patch(\"train_csrnet.validation_loader\", evaluation), \\\n                patch(\"train_csrnet.torch.cuda.is_available\", return_value=False), \\\n                patch(\"train_csrnet.torch.hub.set_dir\"):\n            uninterrupted = Path(directory) / \"uninterrupted\"\n            resumed = Path(directory) / \"resumed\"\n            run(uninterrupted, 2)\n            run(resumed, 1)\n            run(resumed, 2, resumed / \"checkpoints/csrnet_latest.pt\")\n            expected = torch.load(uninterrupted / \"checkpoints/csrnet_latest.pt\", weights_only=True)\n            actual = torch.load(resumed / \"checkpoints/csrnet_latest.pt\", weights_only=True)\n            for key in expected[\"model_state_dict\"]:\n                torch.testing.assert_close(expected[\"model_state_dict\"][key],\n                                           actual[\"model_state_dict\"][key], rtol=0, atol=0)\n            report = json.loads((resumed / \"csrnet_results.json\").read_text())\n            self.assertEqual([entry[\"epoch\"] for entry in report[\"epochs\"]], [1, 2])\n            self.assertIsNone(report[\"test\"])\n            with self.assertRaises(FileExistsError):\n                run(resumed, 3)\n\n            # The original Colab checkpoint has no RNG state or embedded history.\n            legacy = Path(directory) / \"legacy\"\n            run(legacy, 1)\n            best_path = legacy / \"checkpoints/csrnet_best.pt\"\n            old = torch.load(best_path, weights_only=True)\n            for key in (\"history\", \"torch_rng_state\", \"cuda_rng_state_all\",\n                        \"best_epoch\", \"best_validation_mae\"):\n                old.pop(key, None)\n            atomic_save_checkpoint(old, best_path)\n            run(legacy, 2, best_path)\n            history = json.loads((legacy / \"csrnet_results.json\").read_text())[\"epochs\"]\n            self.assertEqual([entry[\"epoch\"] for entry in history], [1, 2])\n\n            # main() installs a file logger; close it before Windows removes the test directory.\n            import logging\n            for handler in list(logging.getLogger().handlers):\n                handler.close()\n                logging.getLogger().removeHandler(handler)\n\n\nif __name__ == \"__main__\":\n    unittest.main()\n"
}


In [ ]:
from datetime import datetime, timezone

EPOCHS = 20
LEARNING_RATE = 1e-5
SKIP_TEST = True  # Set to False to run the final evaluation after training.
ORIGINAL_RUN = DRIVE_ROOT / "csrnet_mall_runs/20260903_010838_885145_UTC"
RUN_DIR = DRIVE_ROOT / "csrnet_mall_runs/20260903_010838_885145_UTC_continued"
assert isinstance(EPOCHS, int) and EPOCHS > 1
assert LEARNING_RATE == 1e-5

# Run the updated source snapshot while keeping the ZIP's verified contents intact.
expected_sources = {
    "csrnet.py", "density_utils.py", "train_csrnet.py",
    "mall_torch_dataset.py", "mall_dataloader.py", "test_csrnet.py",
}
if set(NOTEBOOK_SOURCES) != expected_sources:
    raise RuntimeError("Notebook source snapshot is missing or incomplete.")
source_hashes = {
    name: hashlib.sha256(source.encode("utf-8")).hexdigest()
    for name, source in NOTEBOOK_SOURCES.items()
}
source_id = hashlib.sha256(json.dumps(source_hashes, sort_keys=True).encode()).hexdigest()[:12]
PROJECT_DIR = Path("/content") / f"mall_csrnet_code_{source_id}"
PROJECT_DIR.mkdir(exist_ok=True)
for name, source in NOTEBOOK_SOURCES.items():
    destination = PROJECT_DIR / name
    if destination.exists() and sha256_file(destination) != source_hashes[name]:
        raise RuntimeError(f"An existing staged script differs: {destination}")
    destination.write_text(source, encoding="utf-8")
data_link = PROJECT_DIR / "cv_project"
if not data_link.exists():
    data_link.symlink_to(BUNDLE_PROJECT_DIR / "cv_project", target_is_directory=True)
assert data_link.resolve() == (BUNDLE_PROJECT_DIR / "cv_project").resolve()

subprocess.run(
    [sys.executable, "-m", "unittest", "test_csrnet", "-v"],
    cwd=PROJECT_DIR, check=True,
)

# Preserve the original experiment. Copy only the files needed to continue.
seed_best = ORIGINAL_RUN / "checkpoints/csrnet_best.pt"
seed_report = ORIGINAL_RUN / "csrnet_results.json"
if not seed_best.is_file() or not seed_report.is_file():
    raise FileNotFoundError(f"Original checkpoint/results missing from {ORIGINAL_RUN}")
if RUN_DIR.resolve() == ORIGINAL_RUN.resolve():
    raise RuntimeError("Choose a separate continuation folder to preserve the first run.")
run_checkpoints = RUN_DIR / "checkpoints"
run_checkpoints.mkdir(parents=True, exist_ok=True)
best_path = run_checkpoints / "csrnet_best.pt"
latest_path = run_checkpoints / "csrnet_latest.pt"

def copy_if_absent(source, destination):
    if not destination.exists():
        partial = destination.with_suffix(destination.suffix + ".copying")
        shutil.copyfile(source, partial)
        if sha256_file(partial) != sha256_file(source):
            raise RuntimeError(f"Copy verification failed for {source}")
        partial.replace(destination)

if latest_path.exists() and not best_path.exists():
    raise RuntimeError("Continuation best checkpoint is missing; stop and recover it.")
copy_if_absent(seed_best, best_path)
copy_if_absent(seed_report, RUN_DIR / "csrnet_results.json")
copy_if_absent(seed_report, RUN_DIR / "initial_epoch_results.json")

# The best file may be one epoch newer if interruption happened between the two saves.
candidates = [path for path in (latest_path, best_path) if path.exists()]
def saved_epoch(path):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    return int(saved["epoch"])
RESUME_PATH = max(candidates, key=saved_epoch)
COMPLETED_EPOCH = saved_epoch(RESUME_PATH)
existing_report = json.loads((RUN_DIR / "csrnet_results.json").read_text())
ALREADY_COMPLETE = (
    COMPLETED_EPOCH >= EPOCHS
    and existing_report.get("status") in ("complete", "training_complete")
    and max((item["epoch"] for item in existing_report["epochs"]), default=0) >= EPOCHS
    and (SKIP_TEST or existing_report.get("test") is not None)
)
if COMPLETED_EPOCH >= EPOCHS and not ALREADY_COMPLETE:
    raise RuntimeError("Requested epochs are complete but final results are incomplete. Do not retrain; recover final evaluation first.")

snapshot = RUN_DIR / "source_snapshot"
snapshot.mkdir(exist_ok=True)
for name in NOTEBOOK_SOURCES:
    destination = snapshot / name
    if destination.exists() and sha256_file(destination) != source_hashes[name]:
        raise RuntimeError("This continuation folder belongs to a different code snapshot. Choose a new RUN_DIR.")
    copy_if_absent(PROJECT_DIR / name, destination)
setup = {
    "bundle_sha256": EXPECTED_BUNDLE_SHA256, "source_sha256": source_hashes,
    "parent_run": str(ORIGINAL_RUN), "resume_checkpoint": str(RESUME_PATH),
    "python": sys.version.split()[0], "interpreter": sys.executable,
    "gpu": gpu.name, "gpu_memory_gib": gpu.total_memory / 1024**3,
    "torch": str(torch.__version__), "torchvision": str(torchvision.__version__),
    "cuda": torch.version.cuda, "epochs": EPOCHS,
    "batch_size": 4, "learning_rate": LEARNING_RATE, "skip_test": SKIP_TEST,
}
setup_path = RUN_DIR / f"colab_setup_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}.json"
setup_path.write_text(json.dumps(setup, indent=2), encoding="utf-8")
print("Updated project:", PROJECT_DIR)
print("Resume checkpoint:", RESUME_PATH)
print(f"Completed epoch {COMPLETED_EPOCH}; remaining epochs: {max(0, EPOCHS - COMPLETED_EPOCH)}")
print("Results persist in:", RUN_DIR)
print("Train 1-1400 | Validation 1401-1700 | Test 1701-2000")


## 4. Continue training
The training process uses the kernel's Python interpreter and requires CUDA itself. It restores model weights and Adam state, saves best/latest checkpoints after each completed epoch, and records metrics.

If interrupted, rerun cells in order: setup selects the latest completed checkpoint automatically. An unfinished epoch must be repeated. The original one-epoch checkpoint lacks random-generator state, so its first continuation is valid but cannot reproduce an uninterrupted run exactly.

A completed run is displayed without retraining or testing again. If the GPU runs out of memory, keep the error for review; batch size stays 4.


In [ ]:
import shlex

if not torch.cuda.is_available():
    raise RuntimeError("The GPU is no longer available.")
if "process" in globals() and process.poll() is None:
    raise RuntimeError("Training is already running in this notebook.")
if ALREADY_COMPLETE:
    print("This continuation is already complete; displaying saved results next.")
else:
    command = [
        sys.executable, "-u", "train_csrnet.py",
        "--epochs", str(EPOCHS), "--lr", str(LEARNING_RATE),
        "--output-dir", str(RUN_DIR), "--resume", str(RESUME_PATH),
        "--require-cuda",
    ]
    if SKIP_TEST:
        command.append("--skip-test")
    print("Training command:", shlex.join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Stopped. Completed checkpoints remain in:", RUN_DIR)
        raise
    finally:
        process.stdout.close()
    if return_code != 0:
        raise RuntimeError(f"Training exited with code {return_code}. Review the error before continuing.")
    print("Requested run finished. Reading saved results next.")


## 5. Read the results and three predictions
MAE/RMSE measure errors in **crowd count**, not pixel error. Counts come from the sums of the original ground-truth maps and the predicted maps.

Each saved PNG shows the image, ground-truth density, prediction, and both counts. Density panels share a display scale; enlargement is for viewing only. The test set is **not evaluated again** in this cell.

In [ ]:
from IPython.display import display, Image as NotebookImage

result_path = RUN_DIR / "csrnet_results.json"
if not result_path.is_file():
    raise RuntimeError("No completed results exist yet. Check section 4 before continuing.")
results = json.loads(result_path.read_text())
if results.get("status") not in ("complete", "training_complete"):
    raise RuntimeError("The requested run is not complete yet.")
if max((entry["epoch"] for entry in results["epochs"]), default=0) < EPOCHS:
    raise RuntimeError("The results do not cover the requested epoch count.")
config = results["config"]
assert config["device"] == "cuda"
assert config["batch_size"] == 4 and config["pretrained_vgg16"] is True
assert (RUN_DIR / "checkpoints/csrnet_best.pt").is_file()

print("GPU:", gpu.name)
print("Pretrained VGG16:", config["pretrained_vgg16"])
print("Parameters:", f"{config['parameters']:,}")
print("Batch size:", config["batch_size"], "| learning rate:", config["learning_rate"])
print("Input:", results["preflight"]["input_shape"])
print("Prediction:", results["preflight"]["prediction_shape"])
print("Resized target:", results["preflight"]["resized_target_shape"])
print("Original counts:", results["preflight"]["original_counts"])
print("Resized counts:", results["preflight"]["resized_counts"])
errors = [results["preflight"]["max_count_error"]]
if results.get("test") is not None:
    errors.append(results["test"]["max_count_error"])
for epoch in results["epochs"]:
    errors.extend([epoch["train"]["max_count_error"], epoch["validation"]["max_count_error"]])
    print(f"\nEpoch {epoch['epoch']}")
    print("Training MSE:", epoch["train"]["loss"])
    print("Validation:", epoch["validation"])
    print(f"Epoch time: {epoch['epoch_seconds'] / 60:.2f} minutes")
print("\nMaximum count-preservation error:", max(errors))
print("Best validation epoch:", results["best_epoch"])
print("Final test metrics:", results["test"] if results.get("test") is not None else "Skipped as requested")
print("Checkpoint:", RUN_DIR / "checkpoints/csrnet_best.pt")
print("Full result JSON:", result_path)
print("Prediction folder:", RUN_DIR / "csrnet_predictions")

for path in sorted((RUN_DIR / "csrnet_predictions").glob("test_sample_*.png"))[:3]:
    display(NotebookImage(filename=str(path)))
print("Finished. No further training, model comparison, or dataset is started.")